In [0]:
%sql
DROP TABLE IF EXISTS catalog_ventas.gold.kpi_top_productos;

CREATE OR REPLACE TABLE catalog_ventas.gold.kpi_top_productos
USING DELTA
AS

WITH base AS (
  SELECT
      p.descrip,
      p.categoria,
      SUM(f.total) AS facturacion,
      SUM(f.cant)  AS unidades,
      
      COUNT(DISTINCT f.id_venta) AS tickets

  FROM catalog_ventas.gold.fact_ventas f

  LEFT JOIN catalog_ventas.gold.dim_producto p
    ON f.id_articulo = p.id_articulo 

  GROUP BY p.descrip, p.categoria
),

ranking AS (
  SELECT
  *,
  RANK() OVER (ORDER BY facturacion DESC) AS rnk
  FROM base
)

SELECT * FROM ranking WHERE rnk <= 10

In [0]:
%sql
SELECT * FROM catalog_ventas.gold.kpi_top_productos

In [0]:
%sql

with base AS (
SELECT
CAST(DATE_TRUNC('week', vtafecha) AS DATE) AS semana,
descrip,

  SUM(
    
    CASE 
      WHEN descrip LIKE '%1kilo%' THEN 1000  
      WHEN descrip IN ('1 4 kilo', '1/4kilo') THEN 250 
      WHEN descrip LIKE '%1/2kilo%' THEN 500 
      WHEN descrip LIKE 'tacita 1%' THEN 80
      WHEN descrip LIKE 'tacita 2%' THEN 160
      WHEN descrip LIKE 'tacita 3%' THEN 240
      ELSE 0
    END
  ) AS gramos_granel

FROM catalog_ventas.silver.ventas_clean

WHERE categoria = 'granel'

GROUP BY
CAST(DATE_TRUNC('week', vtafecha) AS DATE),
  descrip
)

select
  semana,
  descrip,
  gramos_granel,
  sum(gramos_granel) OVER (partition by semana) as granel_total 
  FROM base

ORDER BY semana, gramos_granel 


In [0]:
%sql
/*
El granel lidera ampliamente el ranking:
1.	1 kilo
2.	1/4 kilo
3.	1/2 kilo
Estos tres formatos concentran:
•	Mayor facturación
•	Mayor cantidad de tickets
•	Mayor volumen de unidades

•	El kilo es el producto estrella
•	Los bombones, palitos y tortas funcionan como productos de impulso
•	Los formatos familiares acompañan el consumo grupal

*/